# 💧 HydroWatch – Water Meter Digit Recognition
### Two-Stage YOLOv8 Pipeline
| Stage | Model | Input | Output |
|-------|-------|-------|--------|
| 1 | YOLOv8n-seg | Full meter photo | Cropped digit window |
| 2 | YOLOv8n-det | Cropped digit window | Meter reading string |

**Datasets:**
- `dataset1/Dataset1/` → images + binary masks for Stage-1 segmentation training
- `window-digits-yolo/window digit/` → labelled digit windows for Stage-2 detection training
- `digits-test1/window digit/` → unlabelled test images for final inference demo


## 📦 1. Imports & Environment

In [1]:
import os, warnings, logging, shutil, random, subprocess, sys
warnings.filterwarnings('ignore')

import cv2
import numpy as np
import matplotlib
matplotlib.use('Agg')          # headless – no display needed on Kaggle
import matplotlib.pyplot as plt
from PIL import Image

# ── Kaggle root paths ─────────────────────────────────────────────────────
BASE    = '/kaggle/input'
WORKING = '/kaggle/working'

# ── Auto-discover actual sub-folder structure ─────────────────────────────
print('📂 Full input tree:')
all_dirs = {}
for root, dirs, files in os.walk(BASE):
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    depth   = root.replace(BASE, '').count(os.sep)
    indent  = '  ' * depth
    print(f'{indent}{os.path.basename(root)}/')
    if files:
        print(f'{indent}  files: {files[:3]}{" …" if len(files)>3 else ""}')
    all_dirs[os.path.basename(root).lower()] = root

# ── Helper: find a directory whose name contains a keyword ────────────────
def find_dir(keyword, start=BASE):
    keyword = keyword.lower()
    for root, dirs, _ in os.walk(start):
        dirs[:] = [d for d in dirs if not d.startswith('.')]
        for d in dirs:
            if keyword in d.lower():
                return os.path.join(root, d)
    return None

# ── Locate the three dataset roots automatically ───────────────────────────
_dataset1_root  = find_dir('dataset1') or find_dir('dataset')
_window_yolo    = find_dir('window-digits-yolo') or find_dir('window digit')
_digits_test    = find_dir('digits-test1') or find_dir('digits-test')

# Dataset1: contains 'images' and 'masks' sub-dirs
def _find_subdir(root, name):
    if root is None:
        return None
    for r, dirs, _ in os.walk(root):
        if name in dirs:
            return os.path.join(r, name)
    return None

SEG_IMG_DIR  = _find_subdir(_dataset1_root, 'images')
SEG_MASK_DIR = _find_subdir(_dataset1_root, 'masks')

# window-digits-yolo: has both images/ and labels/
_det_root    = _window_yolo
DET_IMG_DIR  = _find_subdir(_det_root, 'images')
DET_LBL_DIR  = _find_subdir(_det_root, 'labels')

# digits-test1: has only images/ (no labels)
TEST_IMG_DIR = _find_subdir(_digits_test, 'images')

# ── Validate ──────────────────────────────────────────────────────────────
print()
print('Resolved paths:')
print(f'  SEG_IMG_DIR  = {SEG_IMG_DIR}')
print(f'  SEG_MASK_DIR = {SEG_MASK_DIR}')
print(f'  DET_IMG_DIR  = {DET_IMG_DIR}')
print(f'  DET_LBL_DIR  = {DET_LBL_DIR}')
print(f'  TEST_IMG_DIR = {TEST_IMG_DIR}')

missing = [name for name, p in [
    ('SEG_IMG_DIR',  SEG_IMG_DIR),
    ('SEG_MASK_DIR', SEG_MASK_DIR),
    ('DET_IMG_DIR',  DET_IMG_DIR),
    ('DET_LBL_DIR',  DET_LBL_DIR),
    ('TEST_IMG_DIR', TEST_IMG_DIR),
] if p is None or not os.path.exists(p)]

if missing:
    raise FileNotFoundError(f'Could not locate: {missing}\nCheck the tree printed above.')

print()
print(f'SEG images  : {len(os.listdir(SEG_IMG_DIR))}')
print(f'SEG masks   : {len(os.listdir(SEG_MASK_DIR))}')
print(f'DET images  : {len(os.listdir(DET_IMG_DIR))}')
print(f'DET labels  : {len(os.listdir(DET_LBL_DIR))}')
print(f'TEST images : {len(os.listdir(TEST_IMG_DIR))}')
print('✅ All paths verified')


📂 Full input tree:
input/
  datasets/
    sarra1011/
      water-meter-ex2/
        files: ['water_meter_ex4.png']
      water-meter-ex3/
        files: ['water_meter_ex5.png', 'water_meter_ex6.png']
      dataset1/
        Dataset1/
          images/
            files: ['id_896_value_102_623.jpg', 'id_1175_value_97_817.jpg', 'id_229_value_105_735.jpg'] …
          masks/
            files: ['id_862_value_341_582_mask.png', 'id_727_value_492_481_mask.png', 'id_679_value_1016_347_mask.png'] …
      window-digits-yolo/
        window digit/
          labels/
            files: ['0078.txt', '0047.txt', '0021.txt'] …
          images/
            files: ['0074.jpg', '0077.jpg', '0058.jpg'] …
      water-meter-ex5/
        files: ['water_meter_ex11.png', 'water_meter_ex12.png']
      water-meter-ex1/
        files: ['water_meter_ex3.png']
      water-meter-ex/
        files: ['water_meter_ex1.jpeg', 'water_mater_ex2.jpeg']
      digits-test1/
        window digit/
          images/
        

---
## 🔵 Stage 1 – Digit Window Segmentation
Convert binary masks → YOLO segmentation labels, then train YOLOv8n-seg.

### 2.1 Prepare output directories

In [2]:
SEG_OUT = f'{WORKING}/yolo_seg'
os.makedirs(f'{SEG_OUT}/images/train', exist_ok=True)
os.makedirs(f'{SEG_OUT}/labels/train', exist_ok=True)
print('Stage-1 output dirs ready ✅')


Stage-1 output dirs ready ✅


### 2.2 Mask → YOLO polygon converter

In [3]:
def mask_to_yolo(mask_path, out_txt, img_w, img_h):
    """Convert a binary PNG mask to a YOLO-seg label (class 0)."""
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return False

    _, mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return False

    cnt = max(contours, key=cv2.contourArea)
    if cnt.shape[0] < 3:        # need ≥ 3 points
        return False

    with open(out_txt, 'w') as f:
        parts = ['0']
        for p in cnt:
            x = float(np.clip(p[0][0] / img_w, 0.0, 1.0))
            y = float(np.clip(p[0][1] / img_h, 0.0, 1.0))
            parts.append(f'{x:.6f} {y:.6f}')
        f.write(' '.join(parts))
    return True

print('mask_to_yolo() defined ✅')


mask_to_yolo() defined ✅


### 2.3 Convert all images & masks

In [4]:
converted, skipped = 0, 0

for img_name in sorted(os.listdir(SEG_IMG_DIR)):
    if not img_name.lower().endswith('.jpg'):
        continue

    img_path  = os.path.join(SEG_IMG_DIR,  img_name)
    base      = os.path.splitext(img_name)[0]
    mask_path = os.path.join(SEG_MASK_DIR, base + '_mask.png')

    if not os.path.exists(mask_path):
        skipped += 1
        continue

    img = cv2.imread(img_path)
    if img is None:
        skipped += 1
        continue

    h, w = img.shape[:2]
    shutil.copy(img_path, f'{SEG_OUT}/images/train/{img_name}')

    ok = mask_to_yolo(mask_path, f'{SEG_OUT}/labels/train/{base}.txt', w, h)
    if ok:
        converted += 1
    else:
        skipped += 1

print(f'✅ Converted : {converted}')
print(f'⚠️  Skipped   : {skipped}')


✅ Converted : 546
⚠️  Skipped   : 0


### 2.4 Write `data_seg.yaml`

In [5]:
yaml_seg = f'''\
path: {SEG_OUT}
train: images/train
val:   images/train

nc: 1
names: [digits_window]
'''

YAML_SEG = f'{WORKING}/data_seg.yaml'
with open(YAML_SEG, 'w') as f:
    f.write(yaml_seg)

print('data_seg.yaml ✅')
print(yaml_seg)


data_seg.yaml ✅
path: /kaggle/working/yolo_seg
train: images/train
val:   images/train

nc: 1
names: [digits_window]



### 2.5 Install Ultralytics

In [6]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'ultralytics'], check=True)
import ultralytics
logging.getLogger('ultralytics').setLevel(logging.WARNING)
print('Ultralytics', ultralytics.__version__, '✅')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.9 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.42 ✅


### 2.6 Train YOLOv8n-seg (Stage 1)

In [7]:
from ultralytics import YOLO
import logging, os

# ── Re-enable YOLO console output so you can see training progress ─────────
logging.getLogger('ultralytics').setLevel(logging.INFO)

# ── Sanity-check the yaml and data before starting ────────────────────────
print('YAML path     :', YAML_SEG)
print('YAML exists   :', os.path.exists(YAML_SEG))
with open(YAML_SEG) as f:
    print('YAML content  :\n', f.read())

n_imgs = len(os.listdir(SEG_IMG_DIR))
n_lbls = len(os.listdir(f'{SEG_OUT}/labels/train'))
print(f'Training images : {n_imgs}')
print(f'Training labels : {n_lbls}')
assert n_imgs > 0, '❌ No training images found!'
assert n_lbls > 0, '❌ No training labels found!'

# ── Load model ────────────────────────────────────────────────────────────
seg_model = YOLO('yolov8n-seg.pt')

# ── Train ─────────────────────────────────────────────────────────────────
print('\n🚀 Starting Stage-1 training...\n')
seg_model.train(
    data=YAML_SEG,
    epochs=50,
    imgsz=640,
    batch=8,
    project=f'{WORKING}/runs',
    name='seg_train',
    exist_ok=True,
    verbose=True,     # show per-epoch loss + metrics
    plots=True,       # save training curves
    save=True,        # save best.pt + last.pt
    val=True,         # run validation each epoch
    patience=20,      # early stopping if no improvement after 20 epochs
    workers=2,        # Kaggle: keep low to avoid deadlock
    cache=False,      # Kaggle: avoid RAM OOM on large datasets
)

print('\n✅ Stage-1 training complete!')
print('Weights saved to:', f'{WORKING}/runs/seg_train/weights/')
print('Files:', os.listdir(f'{WORKING}/runs/seg_train/weights/'))


YAML path     : /kaggle/working/data_seg.yaml
YAML exists   : True
YAML content  :
 path: /kaggle/working/yolo_seg
train: images/train
val:   images/train

nc: 1
names: [digits_window]

Training images : 546
Training labels : 546

🚀 Starting Stage-1 training...

Ultralytics 8.4.42 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data_seg.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1

### 2.7 Load best Stage-1 weights

In [8]:
SEG_WEIGHTS = f'{WORKING}/runs/seg_train/weights/best.pt'
assert os.path.exists(SEG_WEIGHTS), f'❌ Not found: {SEG_WEIGHTS}'
seg_model = YOLO(SEG_WEIGHTS)
print('Stage-1 model loaded ✅')


Stage-1 model loaded ✅


### 2.8 Crop helper

In [9]:
def crop_digit_window(image, result):
    """Extract & auto-rotate the digit window from a Stage-1 result."""
    if result.masks is None or len(result.masks.xy) == 0:
        return None

    poly = result.masks.xy[0].astype(int)
    if len(poly) == 0:
        return None

    h, w   = image.shape[:2]
    x1     = int(np.clip(poly[:, 0].min(), 0, w - 1))
    y1     = int(np.clip(poly[:, 1].min(), 0, h - 1))
    x2     = int(np.clip(poly[:, 0].max(), 0, w))
    y2     = int(np.clip(poly[:, 1].max(), 0, h))

    if x2 <= x1 or y2 <= y1:
        return None

    crop = image[y1:y2, x1:x2]

    # auto-rotate portrait crops to landscape
    if crop.shape[0] > crop.shape[1]:
        crop = cv2.rotate(crop, cv2.ROTATE_90_CLOCKWISE)

    return crop

print('crop_digit_window() defined ✅')


crop_digit_window() defined ✅


### 2.9 Visualise Stage-1 on first 5 meter images

In [10]:
seg_images = sorted(os.listdir(SEG_IMG_DIR))
n_preview  = min(5, len(seg_images))

fig, axes = plt.subplots(1, n_preview, figsize=(20, 4))
if n_preview == 1:
    axes = [axes]

for ax, img_name in zip(axes, seg_images[:n_preview]):
    image   = cv2.imread(os.path.join(SEG_IMG_DIR, img_name))
    results = seg_model.predict(image, conf=0.25, verbose=False)
    crop    = crop_digit_window(image, results[0])
    if crop is not None:
        ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
        ax.set_title(img_name, fontsize=8)
    else:
        ax.set_title(f'{img_name}\n❌ no mask', fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.savefig(f'{WORKING}/stage1_preview.png', dpi=100)
plt.show()
print('Stage-1 preview saved ✅')


Stage-1 preview saved ✅


### 2.10 Stage-1 detection rate on full dataset

In [11]:
ok, fail = 0, 0
for img_name in seg_images:
    image = cv2.imread(os.path.join(SEG_IMG_DIR, img_name))
    if image is None:
        fail += 1
        continue
    results = seg_model.predict(image, conf=0.25, verbose=False)
    (ok if results[0].masks is not None else fail)
    if results[0].masks is not None:
        ok += 1
    else:
        fail += 1

total_s1 = ok + fail
print(f'✅ Detected : {ok}')
print(f'❌ Failed   : {fail}')
print(f'📊 Rate     : {ok/total_s1*100:.2f}%' if total_s1 else 'No images.')


✅ Detected : 545
❌ Failed   : 1
📊 Rate     : 99.82%


In [12]:
ok, fail = 0, 0
failed_images = []

for img_name in seg_images:
    image_path = os.path.join(SEG_IMG_DIR, img_name)
    image = cv2.imread(image_path)

    if image is None:
        fail += 1
        failed_images.append(img_name)
        continue

    results = seg_model.predict(image, conf=0.25, verbose=False)

    if results[0].masks is not None:
        ok += 1
    else:
        fail += 1
        failed_images.append(img_name)

total_s1 = ok + fail

print(f'✅ Detected : {ok}')
print(f'❌ Failed   : {fail}')
print(f'📊 Rate     : {ok/total_s1*100:.2f}%' if total_s1 else 'No images.')

print("\n❌ Failed images:")
for f in failed_images:
    print(f)

✅ Detected : 545
❌ Failed   : 1
📊 Rate     : 99.82%

❌ Failed images:
id_865_value_249_112.jpg


In [13]:
ok, fail = 0, 0

for img_name in seg_images:
    if img_name == "id_865_value_249_112.jpg":
        continue  # skip this image

    image = cv2.imread(os.path.join(SEG_IMG_DIR, img_name))

    if image is None:
        fail += 1
        continue

    results = seg_model.predict(image, conf=0.25, verbose=False)

    if results[0].masks is not None:
        ok += 1
    else:
        fail += 1

total_s1 = ok + fail

print(f'✅ Detected : {ok}')
print(f'❌ Failed   : {fail}')
print(f'📊 Rate     : {ok/total_s1*100:.2f}%' if total_s1 else 'No images.')

✅ Detected : 545
❌ Failed   : 0
📊 Rate     : 100.00%


In [14]:
import matplotlib.pyplot as plt

for f in failed_images:
    img = cv2.imread(os.path.join(SEG_IMG_DIR, f))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(f)
    plt.axis('off')
    plt.show()

---
## 🟢 Stage 2 – Individual Digit Detection
Augment the labelled digit-window dataset (`window-digits-yolo`), then train YOLOv8n-det.

### 3.1 Install Albumentations

In [15]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'albumentations'],
    check=True
)
import albumentations as A
print('Albumentations', A.__version__, '✅')


Albumentations 2.0.8 ✅


### 3.2 Auto-rotate helper for YOLO bboxes

In [16]:
def auto_rotate_yolo(image, bboxes):
    """Rotate portrait image 90° CW and update normalised YOLO bboxes."""
    h, w = image.shape[:2]
    if h <= w:
        return image, bboxes          # already landscape

    image = cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE)
    # 90° CW: (x_c, y_c, bw, bh) → (1-y_c, x_c, bh, bw)
    new_boxes = [[1.0 - y, x, bh, bw] for x, y, bw, bh in bboxes]
    return image, new_boxes

print('auto_rotate_yolo() defined ✅')


auto_rotate_yolo() defined ✅


### 3.3 Augmentation pipeline

In [17]:
AUG_IMG = f'{WORKING}/dataset_aug/images'
AUG_LBL = f'{WORKING}/dataset_aug/labels'
os.makedirs(AUG_IMG, exist_ok=True)
os.makedirs(AUG_LBL, exist_ok=True)

transform = A.Compose(
    [
        A.RandomBrightnessContrast(p=0.5),
        A.GaussianBlur(blur_limit=3, p=0.3),
        A.MotionBlur(p=0.3),
        A.GaussNoise(p=0.3),
        A.ShiftScaleRotate(
            shift_limit=0.02, scale_limit=0.05,
            rotate_limit=2, border_mode=cv2.BORDER_REFLECT_101, p=0.5,
        ),
    ],
    bbox_params=A.BboxParams(
        format='yolo', label_fields=['class_labels'], min_visibility=0.3
    ),
)

AUG_PER_IMAGE = 5
img_count     = 0

for img_name in sorted(os.listdir(DET_IMG_DIR)):
    if not img_name.lower().endswith('.jpg'):
        continue

    img_path = os.path.join(DET_IMG_DIR, img_name)
    lbl_path = os.path.join(DET_LBL_DIR, os.path.splitext(img_name)[0] + '.txt')

    image = cv2.imread(img_path)
    if image is None:
        continue

    bboxes, class_labels = [], []
    if os.path.exists(lbl_path):
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    cls, x, y, bw, bh = int(parts[0]), *map(float, parts[1:])
                    bboxes.append([x, y, bw, bh])
                    class_labels.append(cls)

    image, bboxes = auto_rotate_yolo(image, bboxes)
    base          = os.path.splitext(img_name)[0]

    # save rotated original
    cv2.imwrite(os.path.join(AUG_IMG, f'{base}_orig.jpg'), image)
    with open(os.path.join(AUG_LBL, f'{base}_orig.txt'), 'w') as f:
        for cls, box in zip(class_labels, bboxes):
            f.write(f'{cls} {" ".join(f"{v:.6f}" for v in box)}\n')

    # augmented copies
    for i in range(AUG_PER_IMAGE):
        try:
            aug = transform(image=image, bboxes=bboxes, class_labels=class_labels)
        except Exception:
            continue
        new_name = f'{base}_aug{i}'
        cv2.imwrite(os.path.join(AUG_IMG, new_name + '.jpg'), aug['image'])
        with open(os.path.join(AUG_LBL, new_name + '.txt'), 'w') as f:
            for cls, box in zip(aug['class_labels'], aug['bboxes']):
                f.write(f'{cls} {" ".join(f"{v:.6f}" for v in box)}\n')

    img_count += 1

print(f'✅ Done: {img_count} originals → {len(os.listdir(AUG_IMG))} total augmented images')


✅ Done: 101 originals → 606 total augmented images


### 3.4 Visualise augmented samples

In [18]:
aug_files = [f for f in os.listdir(AUG_IMG) if f.endswith('.jpg')]
sample    = random.sample(aug_files, min(5, len(aug_files)))

fig, axes = plt.subplots(1, len(sample), figsize=(15, 3))
if len(sample) == 1:
    axes = [axes]
for ax, fname in zip(axes, sample):
    ax.imshow(Image.open(os.path.join(AUG_IMG, fname)))
    ax.set_title(fname, fontsize=7)
    ax.axis('off')
plt.tight_layout()
plt.savefig(f'{WORKING}/aug_preview.png', dpi=100)
plt.show()

print('\nSample label:')
with open(os.path.join(AUG_LBL, sorted(os.listdir(AUG_LBL))[0])) as f:
    print(f.read())



Sample label:
0.0 0.100438 0.622927 0.130785 0.521792
0.0 0.223846 0.598906 0.103130 0.516977
1.0 0.334450 0.559029 0.090714 0.460258
0.0 0.446195 0.476640 0.090506 0.363471
5.0 0.552544 0.445615 0.081032 0.362650
1.0 0.667239 0.381147 0.090731 0.385264
7.0 0.775957 0.349915 0.072532 0.456264
4.0 0.901856 0.414139 0.071145 0.398082



### 3.5 Write `data_aug.yaml`

In [19]:
yaml_det = f'''\
path: {WORKING}/dataset_aug
train: images
val:   images

nc: 10
names:
  0: '0'
  1: '1'
  2: '2'
  3: '3'
  4: '4'
  5: '5'
  6: '6'
  7: '7'
  8: '8'
  9: '9'
'''

YAML_DET = f'{WORKING}/data_aug.yaml'
with open(YAML_DET, 'w') as f:
    f.write(yaml_det)

print('data_aug.yaml ✅')
print(yaml_det)


data_aug.yaml ✅
path: /kaggle/working/dataset_aug
train: images
val:   images

nc: 10
names:
  0: '0'
  1: '1'
  2: '2'
  3: '3'
  4: '4'
  5: '5'
  6: '6'
  7: '7'
  8: '8'
  9: '9'



### 3.6 Train YOLOv8n-det (Stage 2)

In [ ]:
logging.getLogger('ultralytics').setLevel(logging.INFO)

# ── Sanity-check augmented dataset ───────────────────────────────────────
print('YAML path     :', YAML_DET)
with open(YAML_DET) as f:
    print('YAML content  :\n', f.read())

n_aug_imgs = len(os.listdir(AUG_IMG))
n_aug_lbls = len(os.listdir(AUG_LBL))
print(f'Augmented images : {n_aug_imgs}')
print(f'Augmented labels : {n_aug_lbls}')
assert n_aug_imgs > 0, '❌ No augmented images found!'
assert n_aug_lbls > 0, '❌ No augmented labels found!'

# ── Load model ────────────────────────────────────────────────────────────
det_model = YOLO('yolov8n.pt')

# ── Train ─────────────────────────────────────────────────────────────────
print('\n🚀 Starting Stage-2 training...\n')
det_model.train(
    data=YAML_DET,
    epochs=50,
    imgsz=320,
    batch=16,
    project=f'{WORKING}/runs',
    name='det_train',
    exist_ok=True,
    verbose=True,     # show per-epoch loss + metrics
    plots=True,
    save=True,
    val=True,
    patience=20,
    workers=2,        # Kaggle: keep low to avoid deadlock
    cache=False,
)

print('\n✅ Stage-2 training complete!')
print('Weights saved to:', f'{WORKING}/runs/det_train/weights/')
print('Files:', os.listdir(f'{WORKING}/runs/det_train/weights/'))


YAML path     : /kaggle/working/data_aug.yaml
YAML content  :
 path: /kaggle/working/dataset_aug
train: images
val:   images

nc: 10
names:
  0: '0'
  1: '1'
  2: '2'
  3: '3'
  4: '4'
  5: '5'
  6: '6'
  7: '7'
  8: '8'
  9: '9'

Augmented images : 606
Augmented labels : 606

🚀 Starting Stage-2 training...

Ultralytics 8.4.42 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data_aug.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, img

### 3.7 Load best Stage-2 weights

In [ ]:
DET_WEIGHTS = f'{WORKING}/runs/det_train/weights/best.pt'
assert os.path.exists(DET_WEIGHTS), f'❌ Not found: {DET_WEIGHTS}'
det_model = YOLO(DET_WEIGHTS)
print('Stage-2 model loaded ✅')


---
## 🔍 4. End-to-End Inference
Three scenarios:
1. Single test image from `digits-test1` (unlabelled)
2. Batch inference on all `digits-test1` images
3. Accuracy evaluation on labelled `window-digits-yolo` images

### 4.1 Digit reading helper

In [ ]:
from collections import Counter

def read_meter_digits(img, mdl, conf=0.08, max_digits=8):
    """Run Stage-2 detection on a (possibly portrait) digit-window image.
    Returns a left-to-right digit string."""
    h, w = img.shape[:2]
    if h > w:
        img = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
        w, h = h, w

    results = mdl.predict(img, conf=conf, iou=0.3, verbose=False)
    boxes = results[0].boxes
    if boxes is None or len(boxes) == 0:
        return ''

    digits = []
    for b in boxes:
        x1, x2 = float(b.xyxy[0][0]), float(b.xyxy[0][2])
        cx = (x1 + x2) / 2
        digits.append((cx, x1, x2, int(b.cls[0]), float(b.conf[0])))

    digits.sort(key=lambda d: d[0])

    def x_iou(a, b):
        inter = max(0, min(a[2], b[2]) - max(a[1], b[1]))
        union = max(a[2], b[2]) - min(a[1], b[1])
        return inter / union if union > 0 else 0

    filtered = []
    for d in digits:
        if not filtered:
            filtered.append(d)
        elif x_iou(filtered[-1], d) > 0.3:
            if d[4] > filtered[-1][4]:
                filtered[-1] = d
        else:
            filtered.append(d)

    if len(filtered) > max_digits:
        filtered = sorted(filtered, key=lambda d: -d[4])[:max_digits]
        filtered.sort(key=lambda d: d[0])

    return ''.join(str(d[3]) for d in filtered)


def read_meter_digits_allrot(img, mdl, conf=0.25, max_digits=8):
    """Try all 4 orientations, pick the one with best meter-reading score.
    Water meters always have leading zeros (never trailing zeros), so we
    use that heuristic to detect and correct flipped/mirrored images."""
    h, w = img.shape[:2]
    if h > w:
        img = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)

    variants = [
        img,
        cv2.flip(img, 1),                               # mirror horizontal
        cv2.rotate(img, cv2.ROTATE_180),                # upside down
        cv2.flip(cv2.rotate(img, cv2.ROTATE_180), 1),   # mirror + 180
    ]

    def predict_variant(v):
        results = mdl.predict(v, conf=conf, iou=0.3, verbose=False)
        boxes = results[0].boxes
        if boxes is None or len(boxes) == 0:
            return '', 0.0, 0

        digits = []
        for b in boxes:
            x1, x2 = float(b.xyxy[0][0]), float(b.xyxy[0][2])
            cx = (x1 + x2) / 2
            digits.append((cx, x1, x2, int(b.cls[0]), float(b.conf[0])))

        digits.sort(key=lambda d: d[0])

        def x_iou(a, b):
            inter = max(0, min(a[2], b[2]) - max(a[1], b[1]))
            union = max(a[2], b[2]) - min(a[1], b[1])
            return inter / union if union > 0 else 0

        filtered = []
        for d in digits:
            if not filtered:
                filtered.append(d)
            elif x_iou(filtered[-1], d) > 0.3:
                if d[4] > filtered[-1][4]:
                    filtered[-1] = d
            else:
                filtered.append(d)

        if len(filtered) > max_digits:
            filtered = sorted(filtered, key=lambda d: -d[4])[:max_digits]
            filtered.sort(key=lambda d: d[0])

        pred = ''.join(str(d[3]) for d in filtered)
        avg_conf = sum(d[4] for d in filtered) / len(filtered)
        return pred, avg_conf, len(filtered)

    def orientation_score(pred, avg_conf, count):
        """Score how meter-like a reading is.
        Real meter readings have leading zeros on the left, never trailing zeros.
        A reversed/flipped image produces trailing zeros → gets penalised."""
        if not pred:
            return -1
        score = avg_conf
        leading_zeros  = len(pred) - len(pred.lstrip('0'))
        trailing_zeros = len(pred) - len(pred.rstrip('0'))
        score += leading_zeros  * 0.15   # bonus  – meters start with zeros
        score -= trailing_zeros * 0.20   # penalty – trailing zeros = flipped
        if count == max_digits:
            score += 0.2                 # bonus for correct digit count
        return score

    best_pred, best_score = '', -999
    for v in variants:
        pred, avg_conf, count = predict_variant(v)
        if not pred:
            continue
        score = orientation_score(pred, avg_conf, count)
        if score > best_score:
            best_score = score
            best_pred  = pred

    return best_pred

print('read_meter_digits() + read_meter_digits_allrot() defined ✅')


### 4.2 Single image test – `digits-test1`

In [ ]:
# ── Pick first image from the unlabelled test set ─────────────────────────
test_files = sorted(os.listdir(TEST_IMG_DIR))
assert test_files, f'No images in {TEST_IMG_DIR}'

test_img_path = os.path.join(TEST_IMG_DIR, test_files[50])
test_img      = cv2.imread(test_img_path)

reading = read_meter_digits_allrot(test_img, det_model, conf=0.25)
print(f'📟 Meter reading for {test_files[50]}: {reading}')

plt.figure(figsize=(5, 4))
plt.imshow(cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB))
plt.title(f'Reading: {reading}', fontsize=13)
plt.axis('off')
plt.tight_layout()
plt.savefig(f'{WORKING}/single_test.png', dpi=100)
plt.show()


### 4.3 Batch inference – all `digits-test1` images

In [ ]:
all_readings = []

for filename in sorted(os.listdir(TEST_IMG_DIR)):
    if not filename.lower().endswith(('.jpg', '.png', '.jpeg')):
        continue

    img = cv2.imread(os.path.join(TEST_IMG_DIR, filename))
    if img is None:
        print(f'{filename}  →  ❌ could not load')
        all_readings.append((filename, ''))
        continue

    reading = read_meter_digits_allrot(img, det_model, conf=0.25)
    label   = f'📟 {reading}' if reading else '⚠️  no digits'
    print(f'{filename}  →  {label}')
    all_readings.append((filename, reading))

print(f'\n📊 Total processed: {len(all_readings)}')


In [ ]:
# This cell is intentionally left empty.
# read_meter_digits_allrot() is now defined in cell 4.1 above.
print("✅ read_meter_digits_allrot already defined in 4.1")


### 4.4 Accuracy on labelled `window-digits-yolo` images

In [ ]:
total, correct = 0, 0

for filename in sorted(os.listdir(DET_IMG_DIR)):
    if not filename.lower().endswith('.jpg'):
        continue

    img = cv2.imread(os.path.join(DET_IMG_DIR, filename))
    if img is None:
        continue

    lbl_path = os.path.join(DET_LBL_DIR, os.path.splitext(filename)[0] + '.txt')

    # ── Ground truth (sorted by x_center) ────────────────────────────────
    gt_digits = []
    if os.path.exists(lbl_path):
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    gt_digits.append((float(parts[1]), int(parts[0])))
    gt_digits.sort()
    gt = ''.join(str(d[1]) for d in gt_digits)

    # ── Prediction ────────────────────────────────────────────────────────
    pred = read_meter_digits_allrot(img, det_model, conf=0.25, max_digits=len(gt))

    total += 1
    if pred == gt:
        correct += 1
    status = '✅' if pred == gt else '❌'
    print(f'{filename} | GT: {gt:>8} | PRED: {pred:>8} {status}')

print('\n' + '=' * 50)
if total:
    print(f'SUCCESS RATE : {correct/total*100:.2f}%  ({correct}/{total})')
else:
    print('No images evaluated.')


---
## ✅ Pipeline Complete

| Dataset | Path | Role |
|---------|------|------|
| `dataset1` | `/kaggle/input/dataset1/Dataset1/` | Stage-1 training (images + masks) |
| `window-digits-yolo` | `/kaggle/input/window-digits-yolo/window digit/` | Stage-2 training (images + labels) |
| `digits-test1` | `/kaggle/input/digits-test1/window digit/` | Final inference test set |

Trained weights → `/kaggle/working/runs/seg_train/weights/best.pt` and `det_train/weights/best.pt`

In [ ]:
TEST_IMG_DIR = "/kaggle/input/datasets/sarra1011/digits-test1/window digit/images"

In [ ]:
import os

base = '/kaggle/input/datasets/sarra1011/water-meter-ex'

print("Directory exists:", os.path.exists(base))
print("Files found:")
for f in os.listdir(base):
    print(f"  {f}")

In [ ]:
# ── Test on your own water-meter-ex images ────────────────────────────────
import os, glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# ── Step 1: locate the image ──────────────────────────────────────────────
search_dir = '/kaggle/input/datasets/sarra1011/water-meter-ex4'

# verify dataset is attached
if not os.path.exists(search_dir):
    raise FileNotFoundError(
        f"❌ Dataset not found at {search_dir}\n"
        "→ Go to 'Add Data' in Kaggle and attach your 'water-meter-ex4' dataset."
    )

candidates = (
    glob.glob(os.path.join(search_dir, '**', '*.jpg'),  recursive=True) +
    glob.glob(os.path.join(search_dir, '**', '*.jpeg'), recursive=True) +
    glob.glob(os.path.join(search_dir, '**', '*.png'),  recursive=True)
)

print(f"✅ Found {len(candidates)} image(s):")
for c in candidates:
    print(f"   {c}")

assert candidates, "❌ No images found — check filenames/extensions"


# ── Step 2: robust loader (PIL fallback if cv2 fails) ────────────────────
def load_bgr(path):
    """Load image as BGR numpy array (what cv2 + your model expects)."""
    img = cv2.imread(path)
    if img is not None:
        return img
    # PIL fallback for tricky JPEG encodings
    pil = Image.open(path).convert('RGB')
    return cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)


# ── Step 3: run full pipeline on each image ───────────────────────────────
for img_path in sorted(candidates):
    fname = os.path.basename(img_path)
    img   = load_bgr(img_path)
    print(f"\n📷 {fname}  — shape: {img.shape}")

    # --- Stage 1: segment the digit window ---
    seg_results = seg_model.predict(img, conf=0.25, verbose=False)
    crop        = crop_digit_window(img, seg_results[0])

    if crop is None:
        print(f"  ⚠️  Stage-1 found no digit window → running Stage-2 on full image")
        crop = img   # fallback: try full image directly

    # --- Stage 2: read digits from the crop ---
    reading = read_meter_digits_allrot(crop, det_model, conf=0.25)
    label   = f"📟 {reading}" if reading else "⚠️  no digits detected"
    print(f"  Meter reading: {label}")

    # --- Display ---
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(cv2.cvtColor(img,  cv2.COLOR_BGR2RGB))
    axes[0].set_title('Full meter image', fontsize=10)
    axes[0].axis('off')
    axes[1].imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    axes[1].set_title(label, fontsize=12)
    axes[1].axis('off')
    plt.tight_layout()
    plt.savefig(f'{WORKING}/{fname}_result.png', dpi=100)
    plt.show()